# TKCE — RF-depth sweep (fight overfitting at the source)

**Idea:** the tree encoding is what lets the network memorize. Shallower RF trees make the encoding **coarser** (broader split bits), which should reduce the neural net's overfitting. This notebook sweeps **RF depth = 3, 4, 6** and reports each.

### How to run
1. **Runtime -> Change runtime type -> GPU** (A100 on Pro+).
2. **Runtime -> Run all.**
3. When cell **4** asks, upload `openml_cache_361070.tar.gz` from your Desktop (OpenML's API is down).

**Runtime ~30-45 min** (3 depths x 4 ablation models x 800 epochs). Each depth's figures go to its own folder; the last cell zips them all for download.

**What to look for** in the `x + tree` rows: does `train_auc` stop snapping to 1.0 at depth 3/4? does the train-test gap shrink? does test AUC hold ~0.69 or nudge toward the tree's 0.708?

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> set Runtime > GPU')

In [ ]:
# 2 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 3 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 4 · upload the dataset cache bundle (OpenML API is down)
from google.colab import files
print('Upload openml_cache_361070.tar.gz from your Desktop:')
files.upload()

In [ ]:
# 5 · extract the cache
import os, glob, tarfile
hits = glob.glob('/content/**/openml_cache_361070.tar.gz', recursive=True)
assert hits, 'Upload the bundle in cell 4 first.'
dst = '/root/.cache/openml/org/openml/www'
os.makedirs(dst, exist_ok=True)
with tarfile.open(hits[0]) as t:
    t.extractall(dst)
print('extracted from', hits[0])
print('tasks:', os.listdir(dst+'/tasks'), '| datasets:', os.listdir(dst+'/datasets'))

In [ ]:
# 6 · verify the dataset loads offline
import openml
t = openml.tasks.get_task(361070, download_splits=False)
d = t.get_dataset(); X, y, *_ = d.get_data(target=d.default_target_attribute)
print('OK cached, no network:', X.shape)

In [ ]:
# 7 · RF-DEPTH SWEEP  (shallower RF = coarser encoding = less overfitting)
# Each depth writes to its own folder results/fusion/depth_<d>.
for d in [3, 4, 6]:
    print(f'\n################  RF depth = {d}  ################\n')
    !python -u run_fusion.py --task 361070 --epochs 800 --fusion tabresnet --rf-depth {d} --rf-min-leaf 20 --out results/fusion/depth_{d} --device auto --ablation

In [ ]:
# 8 · show the figures for each RF depth
from IPython.display import Image, display
import os
for d in [3, 4, 6]:
    curves = f'results/fusion/depth_{d}/fusion_eye_movements_curves.png'
    bar    = f'results/fusion/depth_{d}/fusion_eye_movements.png'
    if os.path.exists(bar):
        print(f'==================  RF depth {d}  ==================')
        display(Image(bar)); display(Image(curves))

In [ ]:
# 9 · download everything (all depths) as one zip
import shutil
from google.colab import files
shutil.make_archive('fusion_depth_sweep', 'zip', 'results/fusion')
files.download('fusion_depth_sweep.zip')